# Radial Importance Analysis

Quantify where the model's predictive information is located by measuring how much attribution falls in the center, middle, and outer image regions.

These radial regions are analysis masks only. They are not dark matter ground truth and are not derived from galaxy brightness thresholds.

## 1. Setup Environment

In [ ]:
# Configure Google Colab environment when needed
import os
import sys
from pathlib import Path

try:
    from google.colab import drive
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if IS_COLAB:
    drive.mount('/content/drive', force_remount=False)
    project_path = Path('/content/drive/MyDrive/xai-dark-matter-localization')
    os.chdir(project_path)
    if str(project_path) not in sys.path:
        sys.path.insert(0, str(project_path))
else:
    project_path = Path.cwd()

print(f'Working directory: {Path.cwd()}')

## 2. Compute Radial Importance Metrics

For each attribution map, the analysis creates three geometric radial masks:

- center: `r < 0.30`
- middle: `0.30 <= r < 0.70`
- outer: `r >= 0.70`

The attribution values are normalized safely. If a map has zero total attribution, all region percentages are set to zero.

In [ ]:
from pathlib import Path
import pandas as pd

from src.analysis.radial_importance import (
    build_radial_importance_table,
    save_radial_importance_plots,
)

xai_dir = Path('results/xai')
summary_csv = xai_dir / 'xai_summary.csv'
analysis_dir = Path('results/analysis')
output_csv = analysis_dir / 'radial_importance.csv'

radial_df = build_radial_importance_table(
    xai_dir=xai_dir,
    summary_csv=summary_csv,
    output_csv=output_csv,
    methods=('gradcam', 'integrated_gradients'),
)

radial_df.head()

## 3. Save Plots

In [ ]:
save_radial_importance_plots(radial_df, output_dir=analysis_dir)

for path in sorted(analysis_dir.glob('*')):
    print(path)

## 4. Mean Importance by Region

This table directly addresses the research question: it summarizes whether the model's predictive information is concentrated near the galaxy center, in intermediate radii, or in the outer image region.

In [ ]:
region_cols = [
    'center_importance_percent',
    'middle_importance_percent',
    'outer_importance_percent',
]

if len(radial_df) > 0:
    display(radial_df.groupby('method')[region_cols].mean())
else:
    print('No radial importance rows found. Generate XAI maps first with notebooks/08_generate_xai_maps.ipynb.')

## 5. Inspect Individual Samples

In [ ]:
if len(radial_df) > 0:
    display(radial_df[[
        'method',
        'subhalo_id',
        'true_halo_mass_log10',
        'pred_halo_mass_log10',
        'center_importance_percent',
        'middle_importance_percent',
        'outer_importance_percent',
        'attribution_path',
    ]].head(20))

## 6. Scientific Interpretation Notes

Use these metrics to compare whether attribution is preferentially central, intermediate, or external across samples and methods. A high percentage in a region means that the trained regressor relied more strongly on pixels in that radial zone for its scalar halo-property prediction. It does not mean that the region contains directly observed dark matter.

## Expected output

The notebook saves `results/analysis/radial_importance.csv`, `mean_importance_by_region.png`, `boxplot_importance_by_region.png`, and `importance_vs_true_halo_mass.png`. The CSV contains `center_importance_percent`, `middle_importance_percent`, and `outer_importance_percent` for each attribution map.